# Load library

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as skl
import anndata as ann
import random, os
from scipy.stats import pearsonr as pr
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score as f1
from sklearn.metrics import precision_recall_curve as prc
from sklearn.metrics import silhouette_score as sil
from sklearn.metrics import auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, average_precision_score
from sklearn.metrics import silhouette_score
from torch_geometric.nn import TransformerConv
from torch_geometric.data import Data
import psutil
import os, sys
import gc
import scipy.sparse as sp
from harmony import harmonize
from tqdm import tqdm
import h5py

In [ ]:
sc.set_figure_params(dpi=200)

# General processing functinos

In [ ]:
def whats_memory_eater():
    # Build reverse map of object id -> variable name from globals
    name_map = {id(obj): name for name, obj in globals().items()}

    # Get all tracked objects
    all_objects = gc.get_objects()

    # Safely get size and match variable name
    sizes = []
    for obj in all_objects:
        try:
            size = sys.getsizeof(obj)
            obj_id = id(obj)
            name = name_map.get(obj_id, None)
            sizes.append((size, type(obj), name, repr(obj)[:100]))
        except Exception:
            continue

    # Sort and print top 10
    sizes.sort(reverse=True, key=lambda x: x[0])

    for size, obj_type, name, preview in sizes[:10]:
        print(f"Size: {size / 1024**3} GB | Type: {obj_type} | Name: {name} | Object: {preview}")


In [ ]:
def memory_usgae():
    gc.collect()
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024**3  # in GB

    print(f"Current memory usage: {memory_gb:.2f} GB")

In [ ]:
def load_and_preprocess_project(base_path, Project_ID, metadata_idx_key='Cell', 
                                Primary_or_Metastatic = 'Primary', remove_doublets = True, doublet_rate=0.06,
                                further_pre = False, file_prefix= None):
    """
    Load and preprocess a single scRNA-seq project with standard filtering and UMAP.
    
    Assumes the base_path contains:
        - One .mtx file (count matrix)
        - One barcodes.csv
        - One features.csv
        - One meta_all.csv
    """
    # Automatically detect files
    files = os.listdir(base_path)
    metadata_file = None
    
    if file_prefix == None:

        mtx_file = [os.path.join(base_path, f) for f in files if f.endswith('.mtx')][0]
        print(mtx_file)
        barcodes_file = [os.path.join(base_path, f) for f in files if 'barcode' in f][0]
        print(barcodes_file)
        try:
            features_file = [os.path.join(base_path, f) for f in files if 'feature' in f][0]
        except:
            features_file = [os.path.join(base_path, f) for f in files if 'genes' in f][0]
        print(features_file)
        metadata_file = [os.path.join(base_path, f) for f in files if 'meta' in f][0]
        print(metadata_file)
    else:
        for f in files:
            if not f.startswith(file_prefix):
                continue
            if f.endswith('mtx'):
                mtx_file = os.path.join(base_path, f)
                print(mtx_file)
            elif 'barcode' in f:
                barcodes_file = os.path.join(base_path, f)
                print(barcodes_file)
            elif 'feature' in f:
                features_file = os.path.join(base_path, f)
                print(features_file)
            elif 'genes' in f:
                features_file = os.path.join(base_path, f)
                print(features_file)
            elif 'meta' in f:
                metadata_file = os.path.join(base_path, f)
                print(metadata_file)
            else:
                continue
    # print(metadata_file)
    print(f"Loading: {mtx_file}")

    # Load matrix
    adata = sc.read_mtx(mtx_file)
    adata = adata.transpose()  # Important: make cells as rows, genes as columns

    # Load barcodes and features
    if barcodes_file.endswith('tsv'):
        barcodes = pd.read_csv(barcodes_file, sep='\t', header=None)  # no header=None here
    else:
        barcodes = pd.read_csv(barcodes_file)  # no header=None here
    display(barcodes)
    
    if features_file.endswith('tsv'):
        genes = pd.read_csv(features_file, sep='\t', header=None)  # no header=None here
    else:
        genes = pd.read_csv(features_file)  # no header=None here
    display(genes)

    # Assign barcodes and gene names (convert to string)
    if barcodes.shape[1] > 1:
        adata.obs_names = barcodes.iloc[:, 1].astype(str).values
    else:
        adata.obs_names = barcodes.iloc[:, 0].astype(str).values
    
    if genes.shape[1] > 1:
        adata.var_names = genes.iloc[:, 1].astype(str).values
    else:
        adata.var_names = genes.iloc[:, 0].astype(str).values
    # adata.var_names = genes.iloc[:, 0].astype(str).values
    display(adata.to_df())

    # Load and merge metadata
    # if metadata_file
    try:
        if metadata_file.endswith('tsv'):
            metadata = pd.read_csv(metadata_file, sep='\t')
        elif metadata_file.endswith('csv'):
            metadata = pd.read_csv(metadata_file)
        metadata.index = metadata[metadata_idx_key]
        adata.obs = adata.obs.join(metadata, how='left')
    except:
        pass         
    
    # DOUBLET DETECTION (before other filtering)
    if remove_doublets:
        print(f'Running doublet detection on {adata.n_obs} cells...')
                
        sc.external.pp.scrublet(adata, expected_doublet_rate=0.06)
        
        n_cells = adata.n_obs
        n_doublets = adata.obs['predicted_doublet'].sum()
        print(f'  Detected {n_doublets} doublets ({n_doublets/n_cells*100:.1f}%)')
        
        adata = adata[~adata.obs['predicted_doublet']].copy()
        print(f'  After doublet removal: {adata.n_obs} cells')

    # Calculate QC metrics
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

    # Standard cell filtering
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) & 
                  (adata.obs['n_genes_by_counts'] <= 5000) & 
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    adata.obs['Project_ID'] = Project_ID
    adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    # Normalize and log transform
    adata.raw = adata.copy()
    
    if further_pre:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

        # Highly variable genes
        # sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

        # Keep only HVGs
        # adata = adata[:, adata.var.highly_variable]

        # Scale
        sc.pp.scale(adata, max_value=10)

        # PCA
        sc.tl.pca(adata, svd_solver='arpack')

        # Neighbors
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)

        # UMAP
        sc.tl.umap(adata)

    print(f"Finished processing {Project_ID}. Shape: {adata.shape}")

    return adata


In [ ]:
def filter_and_recompute(adata, celltype_col, celltypes_to_keep, further_pre = False):
    """
    Filters an AnnData object to keep only specified cell types, 
    then recalculates PCA, neighbors, and UMAP.

    Parameters:
    - adata: AnnData object
    - celltype_col: str, the column in adata.obs containing cell type annotations
    - celltypes_to_keep: list of str, the cell types you want to keep

    Returns:
    - filtered and recalculated AnnData object
    """
    # Step 1: Filter cells
    print(f"Original shape: {adata.shape}")
    adata_filtered = adata[adata.obs[celltype_col].isin(set(celltypes_to_keep))].copy()
    print(f"Filtered shape: {adata_filtered.shape}")

    # Step 2: Recalculate PCA and UMAP
    # (Assumes data is already normalized and scaled)
    if further_pre:
        sc.tl.pca(adata_filtered, svd_solver='arpack')
        sc.pp.neighbors(adata_filtered, n_neighbors=15, n_pcs=40)
        sc.tl.umap(adata_filtered)

    print("Recalculated PCA and UMAP.")
    return adata_filtered

In [ ]:
def reprocess_from_raw_layer(adata, Project_ID, Primary_or_Metastatic='Primary', 
                             further_pre=False, remove_doublets=True, doublet_rate=0.06):
    """
    Reprocess a Scanpy AnnData object using its raw layer (e.g., from a published .h5ad).
    This includes doublet removal, normalization, HVG selection, PCA, neighbors, and UMAP.
    
    Parameters:
    - adata: AnnData object, must have .raw set
    - Project_ID: Project identifier
    - Primary_or_Metastatic: Sample type ('Primary' or 'Metastatic')
    - further_pre: Whether to do full preprocessing (normalization, PCA, UMAP)
    - remove_doublets: Whether to run doublet detection and filtering
    - doublet_rate: Expected doublet rate (default 0.06 = 6%)
    
    Returns:
    - Processed AnnData object (modifies in place)
    """
    # Check if raw exists
    if adata.raw is None:
        raise ValueError("AnnData object has no .raw attribute. Cannot proceed with reprocessing.")
    
    # Extract raw counts
    adata.X = adata.raw.X.copy()
    adata.var = adata.raw.var.copy()
    adata.var_names = adata.raw.var_names.copy()
    
    # DOUBLET DETECTION (before other filtering)
    if remove_doublets:
        print(f'Running doublet detection on {adata.n_obs} cells...')
                
        sc.external.pp.scrublet(adata, expected_doublet_rate=0.06)
        
        n_cells = adata.n_obs
        n_doublets = adata.obs['predicted_doublet'].sum()
        print(f'  Detected {n_doublets} doublets ({n_doublets/n_cells*100:.1f}%)')
        
        adata = adata[~adata.obs['predicted_doublet']].copy()
        print(f'  After doublet removal: {adata.n_obs} cells')
    
    # Recalculate mitochondrial content
    adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    print('Standard filtering...')
    # Standard filtering (optional)
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) &
                  (adata.obs['n_genes_by_counts'] <= 5000) &
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    # Add metadata
    adata.obs['Project_ID'] = Project_ID
    adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    if further_pre:
        print('Normalizing...')
        # Normalize and log transform
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        
        print('Scaling...')
        sc.pp.scale(adata, max_value=10)
        
        print('Computing PCA...')
        sc.tl.pca(adata, svd_solver='arpack')
        
        print('Computing neighbors...')
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
        
        print('Computing UMAP...')
        sc.tl.umap(adata)
    
    print(f"Reprocessed dataset. Final shape: {adata.shape}")
    return adata

In [ ]:
def reprocess_all(adata, further_pre = True):
    """
    Reprocess a Scanpy AnnData object using its raw layer (e.g., from a published .h5ad).
    This includes normalization, HVG selection, PCA, neighbors, and UMAP.

    Parameters:
    - adata: AnnData object, must have .raw set

    Returns:
    - Processed AnnData object (modifies in place)
    """

    # Check if raw exists
    if adata.raw is None:
        raise ValueError("AnnData object has no .raw attribute. Cannot proceed with reprocessing.")

    # Extract raw counts
    adata.X = adata.raw.X.copy()
    adata.var = adata.raw.var.copy()
    adata.var_names = adata.raw.var_names.copy()

    # Recalculate mitochondrial content
    adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    print('Standard filtering...')
    # Standard filtering (optional)
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) &
                  (adata.obs['n_genes_by_counts'] <= 5000) &
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    # adata.obs['Project_ID'] = Project_ID
    # adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    if further_pre:
        # Normalize and log transform
        print('Normalizing...')
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

        # HVG selection
        # sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
        # adata = adata[:, adata.var.highly_variable]
        '''
        sc.pp.highly_variable_genes(
            adata,
            flavor="seurat_v3",  # best for batch-aware HVG selection
            n_top_genes=2000,
            batch_key="Final_sample_id"  # or whatever your batch label column is
        )
        '''
        
        # adata = adata[:, adata.var.highly_variable].copy()

        # Scale
        # print('Scaling...')
        # sc.pp.scale(adata, max_value=10)
        print('Computing PCA...')
        # PCA, neighbors, UMAP
        sc.tl.pca(adata, zero_center=False)
        
        print('Computing neighbors...')
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
        
        print('Computing UMAP...')
        sc.tl.umap(adata)

    print(f"Reprocessed dataset. Final shape: {adata.shape}")
    return adata


In [ ]:
def reset_plot():
    # After running scrublet, reset the display settings
    import matplotlib.pyplot as plt
    import matplotlib
    %matplotlib inline

    # Reset matplotlib backend
    matplotlib.use('module://matplotlib_inline.backend_inline')

    # Reset scanpy settings
    import scanpy as sc
    sc.settings.autoshow = True
    sc.settings.set_figure_params(dpi=100, facecolor='white')

# Colorectal Cancer (COAD)

## A pan-cancer blueprint of the heterogeneous tumor microenvironment revealed by single-cell profiling


Paper: https://www.nature.com/articles/s41422-020-0355-0#Fig3

Data downloaded from: https://lambrechtslab.sites.vib.be/en/pan-cancer-blueprint-tumour-microenvironment-0

Link: 
- Matrix: https://lambrechtslab.sites.vib.be/en/pan-cancer-blueprint-tumour-microenvironment-0 (Colorectal cancer - Counts Matrix)
- Patient metadata: https://static-content.springer.com/esm/art%3A10.1038%2Fs41422-020-0355-0/MediaObjects/41422_2020_355_MOESM13_ESM.pdf
- Sequecing quality: https://www.nature.com/
https://static-content.springer.com/esm/art%3A10.1038%2Fs41422-020-0355-0/MediaObjects/41422_2020_355_MOESM14_ESM.pdf

In [ ]:
memory_usgae()

In [ ]:
# Set the project directory
project_dir = "../../Data/COAD/2098-Colorectalcancer/"  # <- change this for each project

# Load and preprocess
ad = load_and_preprocess_project(project_dir, 
                                 Project_ID='2098-Colorectalcancer', 
                                 Primary_or_Metastatic='Primary',
                                 further_pre=True)

In [ ]:
ad.obs['PatientNumber'] = ad.obs['PatientNumber'].astype(str)

In [ ]:
ad = filter_and_recompute(adata=ad, 
                          celltype_col='CellType', 
                          celltypes_to_keep=['Cancer'],
                          further_pre=True)
ad

In [ ]:
patient_cell_number = pd.read_csv("../../Data/COAD/2098-Colorectalcancer/2099-Colorectalcancer_metadata.csv")['PatientNumber'].value_counts()
patient_cell_number = patient_cell_number.to_dict()
patient_cell_number

In [ ]:
pd.read_csv('../../Data/BRCA/2102-Breastcancer/Sequencing_quality_S2.txt', sep='\t')['Cancer type'].value_counts()

In [ ]:
patient_seuqncing_meta_df = pd.read_csv('../../Data/BRCA/2102-Breastcancer/Sequencing_quality_S2.txt', sep='\t')
patient_seuqncing_meta_df = patient_seuqncing_meta_df[patient_seuqncing_meta_df['Cancer type'] == 'CRC']
patient_seuqncing_meta_df

In [ ]:
# Group by 'Patient number' and sum the 'Cells' column
cells_to_lc_label = patient_seuqncing_meta_df.groupby("Patient number")["Cells"].sum().to_dict()

# Flip the dict so it's {cell_sum: patient_id}
cells_to_lc_label = {v: k for k, v in cells_to_lc_label.items()}
cells_to_lc_label


In [ ]:
patient_number_to_LC_id = dict()
for patient_number in patient_cell_number.keys():
    # print(patient_number)
    try:
        patient_number_to_LC_id[str(patient_number)] = cells_to_lc_label[patient_cell_number[patient_number]]
    except:
        print(patient_number)
patient_number_to_LC_id

In [ ]:
ad.obs['BC_PatientID'] = ad.obs['PatientNumber'].map(patient_number_to_LC_id)
ad.obs

In [ ]:
patient_meta_df = pd.read_csv('../../Data/BRCA/2102-Breastcancer/Patient_metadata_S1.txt', sep='\t')
patient_meta_df = patient_meta_df[patient_meta_df['Tumor_type'] == 'CRC']
meta_subset = patient_meta_df
meta_subset

In [ ]:
ad.obs = ad.obs.merge(meta_subset, left_on='BC_PatientID', right_on='Patient_number', how='left')
ad.obs

In [ ]:
# Ensure TNM is string
ad.obs['TNM'] = ad.obs['TNM'].astype(str)

# If Primary_or_Metastatic is categorical, add new category first
if pd.api.types.is_categorical_dtype(ad.obs['Primary_or_Metastatic']):
    ad.obs['Primary_or_Metastatic'] = ad.obs['Primary_or_Metastatic'].cat.add_categories(['Metastatic'])

# Now assign "Metastatic" to rows where TNM contains "M1"
ad.obs.loc[ad.obs['TNM'].str.contains('M1', na=False), 'Primary_or_Metastatic'] = 'Metastatic'


In [ ]:
ad.obs['Primary_or_Metastatic'].value_counts()

In [ ]:
ad.obs['Final_cancer_type'] = 'Colorectal Cancer'
ad.obs['Final_histological_subtype'] = ad.obs.Pathological_subtype
ad.obs['Final_molecular_subtype'] = ad.obs.Molecular_status
ad.obs['Final_tissue'] = 'Colon'
ad.obs['Final_sample_id'] = ad.obs['BC_PatientID']

In [ ]:
ad

In [ ]:
# add more clinical information
ad.obs['Final_patient_age'] = ad.obs['Age_range']
ad.obs['Final_patient_stage'] = ad.obs['TNM']
ad.obs['Final_patient_treatment'] = 'Naïve'

In [ ]:
ad.raw.shape

In [ ]:
ad

In [ ]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/2098-Colorectalcancer.COAD.h5ad', compression='gzip')

## Single-cell and spatial transcriptome analysis reveals the cellular heterogeneity of liver metastatic colorectal cancer


Paper: https://www.science.org/doi/full/10.1126/sciadv.adf5464?rfr_dat=cr_pub++0pubmed&url_ver=Z39.88-2003&rfr_id=ori%3Arid%3Acrossref.org#sec-4

Data downloaded from: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE225857

Data File: 
- GSM7058755_non_immune_counts.txt.gz
- GSM7058755_non_immune_meta.txt.gz


In [ ]:
file_path = '../../Data/COAD/GSE225857/GSM7058755_non_immune_counts.txt'

In [ ]:
# Step 1: Read just the first line to get cell names
with open(file_path) as f:
    header = f.readline().strip().split('\t')
cell_names = header[1:]  # skip the gene column
print(cell_names[:10])
# Step 2: Read file in chunks and store rows
gene_names = []
data = []

# count = 0
with open(file_path) as f:
    next(f)  # skip header
    for line in tqdm(f, desc="Reading rows"):
        parts = line.strip().split('\t')
        gene = parts[0]
        counts = np.array(parts[1:], dtype=np.float32)
        gene_names.append(gene)
        data.append(counts)
print(gene_names[:10])

# Step 3: Stack into matrix and transpose
dense_matrix = np.vstack(data)  # shape: (genes, cells)
transposed = sp.csr_matrix(dense_matrix.T)  # shape: (cells, genes)

# Step 4: Create AnnData
ad = ann.AnnData(X=transposed,
                   obs=pd.DataFrame(index=cell_names),
                   var=pd.DataFrame(index=gene_names))
ad

In [ ]:
ad.obs_names = ad.obs_names.str.replace('"', '', regex=False).str.replace('.', '-')
ad.var_names = ad.var_names.str.replace('"', '', regex=False)

In [ ]:
ad.to_df()

In [ ]:
metadata_df = pd.read_csv('../../Data/COAD/GSE225857/GSM7058755_non_immune_meta.txt', sep='\t', index_col=0)
metadata_df

In [ ]:
ad.obs = metadata_df.loc[ad.obs.index]
ad

In [ ]:
ad.raw = ad

In [ ]:
# re-process the adata
ad = reprocess_from_raw_layer(ad, 
                              Project_ID='GSE225857', 
                              Primary_or_Metastatic='Metastatic',
                              further_pre=True)

In [ ]:
for obs in ['patients', 'organs', 'cluster']:
    sc.pl.umap(ad, color=obs)

In [ ]:
tumor_types = []
for i in ad.obs.cluster.value_counts().index:
    if i.startswith('Tu'):
        tumor_types.append(i)
tumor_types

In [ ]:
ad = filter_and_recompute(adata=ad, 
                          celltype_col='cluster', 
                          celltypes_to_keep=tumor_types,
                          further_pre=True)

In [ ]:
for obs in ['patients', 'organs', 'cluster']:
    sc.pl.umap(ad, color=obs)

In [ ]:
ad

In [ ]:
new_tissues = []
for tissue in ad.obs.organs:
    if tissue == 'CCT':
        new_tissues.append('Colon')
    elif tissue == 'LCT':
        new_tissues.append('Liver')
    else:
        print('Wrong')

In [ ]:
ad.obs['Final_cancer_type'] = 'Colorectal Cancer'
ad.obs['Final_histological_subtype'] = 'COAD: Unspecified'
ad.obs['Final_molecular_subtype'] = 'COAD: Unspecified'
ad.obs['Final_tissue'] = new_tissues
ad.obs['Final_sample_id'] = ad.obs['patients']

In [ ]:
# Clinical + Treatment metadata (merged)
patient_metadata = {
    "s0107": {"Age": 75, "Gender": "Male", "Primary site": "Left colon", "Tissues": ["CN", "CC", "LN", "LM", "PB"], "Regimen": "FOLFOX", "Cycles": 3},
    "s0115": {"Age": 80, "Gender": "Male", "Primary site": "Left colon", "Tissues": ["CN", "CC", "LN", "LM", "PB"], "Regimen": "FOLFOX", "Cycles": 4},
    "s0813": {"Age": 58, "Gender": "Female", "Primary site": "Right colon", "Tissues": ["CN", "CC", "LN", "LM", "PB"], "Regimen": "FOLFOXIRI", "Cycles": 5},
    "s0920": {"Age": 64, "Gender": "Male", "Primary site": "Left colon", "Tissues": ["CC", "LN", "LM", "PB"], "Regimen": "FOLFOX", "Cycles": 5},
    "s1125": {"Age": 41, "Gender": "Female", "Primary site": "Left colon", "Tissues": ["CN", "CC", "LN", "LM"], "Regimen": "FOLFOXIRI+Bevacizumab", "Cycles": 8},
    "s1231": {"Age": 58, "Gender": "Male", "Primary site": "Left colon", "Tissues": ["CN", "CC", "LN", "LM", "PB"], "Regimen": "FOLFOX", "Cycles": 3}
}

In [ ]:
# Extract dictionaries from the metadata you want to add
age_map = {k: v["Age"] for k, v in patient_metadata.items()}
# stage_map = {k: v.get("Cycles", None) for k, v in patient_metadata.items()}  # or use another field if you meant clinical stage
treatment_map = {k: v["Regimen"] for k, v in patient_metadata.items()}


In [ ]:
# add more clinical information
ad.obs['Final_patient_age'] = ad.obs['patients'].map(age_map)
ad.obs['Final_patient_stage'] = 'Unknown'
ad.obs['Final_patient_treatment'] = ad.obs['patients'].map(treatment_map)

In [ ]:
for obs in ['Final_patient_age', 'Final_patient_treatment']:
    sc.pl.umap(ad, color=obs)

In [ ]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/GSE225857.COAD.h5ad', compression='gzip')

## Spatially organized multicellular immune hubs in human colorectal cancer


Paper: https://www.cell.com/cell/fulltext/S0092-8674(21)00945-4

Data downloaded from: 
- https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE178341
- https://singlecell.broadinstitute.org/single_cell/study/SCP1162/human-colon-cancer-atlas-c295#study-download

Files: 
- Matrix:https://ftp.ncbi.nlm.nih.gov/geo/series/GSE178nnn/GSE178341/suppl/GSE178341%5Fcrc10x%5Ffull%5Fc295v4%5Fsubmit.h5 
- Annotation: 
-- https://singlecell.broadinstitute.org/single_cell/data/public/SCP1162/human-colon-cancer-atlas-c295?filename=metatable_v3_fix_v3.tsv
-- https://singlecell.broadinstitute.org/single_cell/data/public/SCP1162/human-colon-cancer-atlas-c295?filename=crc10x_tSNE_cl_global.tsv


In [ ]:
# Step 1: Load from the .h5 file
with h5py.File('../../Data/COAD/GSE178341/GSE178341_crc10x_full_c295v4_submit.h5', 'r') as f:
    mat = f['matrix']
    
    # Sparse matrix components
    data = mat['data'][:]
    indices = mat['indices'][:]
    indptr = mat['indptr'][:]
    shape = tuple(mat['shape'])  # e.g. (genes, cells)

    # Cell barcodes and feature (gene) names
    barcodes = mat['barcodes'][:].astype(str)
    features = mat['features']['name'][:].astype(str)

# Step 2: Build sparse matrix
X = sp.csc_matrix((data, indices, indptr), shape=shape)

# Step 3: Create AnnData object
# By convention: cells = obs = rows, genes = vars = columns → transpose
ad = ann.AnnData(X.T)
ad.var_names = features
ad.obs_names = barcodes
ad

In [ ]:
memory_usgae()
del X, data
memory_usgae()

In [ ]:
ad.raw = ad

In [ ]:
metadata_df = pd.read_csv('../../Data/COAD/GSE178341/metatable_v3_fix_v3.tsv', sep='\t', index_col=0)
metadata_df

In [ ]:
ad.obs = metadata_df.loc[ad.obs.index]
ad

In [ ]:
metadata_df = pd.read_csv('../../Data/COAD/GSE178341/crc10x_tSNE_cl_global.txt', sep='\t', index_col=0)

In [ ]:
ad.obs['ClusterMidway'] = metadata_df.loc[ad.obs.index]['ClusterMidway']

In [ ]:
ad = ad[ad.obs['ClusterMidway'].isin(set(['EpiT']))].copy()

In [ ]:
ad

In [ ]:
# re-process the adata
ad = reprocess_from_raw_layer(ad, 
                              Project_ID='GSE178341', 
                              Primary_or_Metastatic='Metastatic',
                              further_pre=True)

In [ ]:
reset_plot()
for obs in ['LymphNodeStatus', 'disease', 'species__ontology_label']:
    sc.pl.umap(ad, color=obs)

In [ ]:
ad

In [ ]:
patient_clinical_df = pd.read_csv('../../Data/COAD/GSE178341/Petient_clinical.txt', sep='\t')
patient_clinical_df

In [ ]:
# Step 1: Reset index for merging, but save cell IDs
merged = ad.obs.reset_index().merge(
    patient_clinical_df,
    left_on='PatientTypeID',
    right_on='PatientBarcode_SpecimenType',
    how='left'
)

# Step 2: Restore original index (cell barcodes)
merged = merged.set_index('index')

# Step 3: Assign back
ad.obs = merged

In [ ]:
ad.obs['Project_ID'] = 'GSE178341'
ad.obs['Primary_or_Metastatic'] = ad.obs["Metastasis stage (on resection specimen path report)"].astype(str).apply(
    lambda x: "Metastatic" if "M1" in x else "M0"
)

In [ ]:
ad.obs['Final_cancer_type'] = 'Colorectal Cancer'
ad.obs['Final_histological_subtype'] = ['COAD: '+i for i in ad.obs['HistologicTypeSimple']]
ad.obs['Final_molecular_subtype'] = ['COAD: '+i for i in ad.obs['MMRStatus']]
ad.obs['Final_tissue'] = 'Colon'
ad.obs['Final_sample_id'] = ad.obs['donor_id']

In [ ]:
ad.raw.shape

In [ ]:
all_qc_obs = []
for col in ad.obs.columns:
    if col.startswith('qc_'):
        all_qc_obs.append(col)
all_qc_obs

In [ ]:
ad.obs = ad.obs.drop(columns=all_qc_obs)


In [ ]:
# add more clinical information
ad.obs['Final_patient_age'] = ad.obs.Age
ad.obs['Final_patient_stage'] = ad.obs['Tumor Stage Raw (on resection specimen path report)']
ad.obs['Final_patient_treatment'] = 'Naïve'

In [ ]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/GSE178341.COAD.h5ad', compression='gzip')

## Lineage-dependent gene expression programs influence the immune landscape of colorectal cancer.

Paper: https://www.nature.com/articles/s41588-020-0636-z#Fig2

Data downloaded from: 
- https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE132465

Files: 
- Matrix:https://ftp.ncbi.nlm.nih.gov/geo/series/GSE132nnn/GSE132465/suppl/GSE132465%5FGEO%5Fprocessed%5FCRC%5F10X%5Fraw%5FUMI%5Fcount%5Fmatrix.txt.gz
- Annotation: https://ftp.ncbi.nlm.nih.gov/geo/series/GSE132nnn/GSE132465/suppl/GSE132465%5FGEO%5Fprocessed%5FCRC%5F10X%5Fcell%5Fannotation.txt.gz
- Additional annotation: https://www.dropbox.com/scl/fi/go3m1x3j3gtmhmjexbvk6/Meta-data_Lee2020_Colorectal.tar.gz?rlkey=4p2qlwuz3ay3oztnun3b9kgfu&dl=1


In [ ]:
file_path = '../../Data/COAD/GSE132465/GSE132465_GEO_processed_CRC_10X_raw_UMI_count_matrix.txt'

In [ ]:
# Step 1: Read just the first line to get cell names
with open(file_path) as f:
    header = f.readline().strip().split('\t')
cell_names = header[1:]  # skip the gene column
print(cell_names[:10])
# Step 2: Read file in chunks and store rows
gene_names = []
data = []

# count = 0
with open(file_path) as f:
    next(f)  # skip header
    for line in tqdm(f, desc="Reading rows"):
        parts = line.strip().split('\t')
        gene = parts[0]
        counts = np.array(parts[1:], dtype=np.float32)
        gene_names.append(gene)
        data.append(counts)
print(gene_names[:10])


In [ ]:

# Step 3: Stack into matrix and transpose
dense_matrix = np.vstack(data)  # shape: (genes, cells)
transposed = sp.csr_matrix(dense_matrix.T)  # shape: (cells, genes)

# Step 4: Create AnnData
ad = ann.AnnData(X=transposed,
                   obs=pd.DataFrame(index=cell_names),
                   var=pd.DataFrame(index=gene_names))
ad

In [ ]:
meta_df = pd.read_csv('../../Data/COAD/GSE132465/GSE132465_GEO_processed_CRC_10X_cell_annotation.txt', sep='\t', index_col=0)
meta_df

In [ ]:
ad.obs = meta_df.loc[ad.to_df().index]

In [ ]:
ad = ad[ad.obs['Class'] == 'Tumor']
ad

In [ ]:
ad.raw = ad

In [ ]:
additional_meta_df = pd.read_csv('../../Data/COAD/GSE132465/Data_Lee2020_Colorectal/Cells.csv', index_col=0)
additional_meta_df

In [ ]:
shared_cells = list(set(additional_meta_df.index).intersection(set(ad.to_df().index)))
shared_cells.sort()
ad = ad[shared_cells]
ad

In [ ]:
ad.obs = additional_meta_df.loc[shared_cells]

In [ ]:
# re-process the adata
ad = reprocess_from_raw_layer(ad, 
                              Project_ID='GSE132465', 
                              Primary_or_Metastatic='Metastatic',
                              further_pre=False)

In [ ]:
ad = filter_and_recompute(adata=ad, 
                          celltype_col='cell_type', 
                          celltypes_to_keep=['Malignant'],
                          further_pre=True)
ad

In [ ]:
reset_plot()
sc.pl.umap(ad, color=['sample'])

In [ ]:
patient_clinical_df = pd.read_csv('../../Data/COAD/GSE132465/patient_clinical.txt', sep='\t')
patient_clinical_df

In [ ]:
merged = ad.obs.reset_index().merge(
    patient_clinical_df,
    left_on='sample',
    right_on='Patient',
    how='left'
)

# Step 2: Restore original index (cell barcodes)
merged = merged.set_index('cell_name')

# Step 3: Assign back
ad.obs = merged
ad.obs

In [ ]:
primary_or_metastatic = []
for i in ad.obs['TNM stage']:
    if i.__contains__('M1'):
        primary_or_metastatic.append('Metastatic')
    else:
        primary_or_metastatic.append('Primary')

In [ ]:
# add more clinical information
ad.obs['Project_ID'] = 'GSE132465'
ad.obs['Primary_or_Metastatic'] = primary_or_metastatic
ad.obs['Final_cancer_type'] = 'Colorectal Cancer'
ad.obs['Final_histological_subtype'] = 'Unknown'
ad.obs['Final_molecular_subtype'] = 'Unknown'
ad.obs['Final_tissue'] = 'Colon'
ad.obs['Final_sample_id'] = ad.obs['sample']
ad.obs['Final_patient_age'] = ad.obs['Age']
ad.obs['Final_patient_stage'] = ad.obs['Stage']
ad.obs['Final_patient_treatment'] = 'Naive'

In [ ]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/GSE132465.COAD.h5ad', compression='gzip')

## Integrate the data

In [ ]:
data_dir = '../../Data/Cancer_cell_data_reprocessed/'
all_h5_files = os.listdir(data_dir)
all_h5_files.sort()

all_h5_files

In [ ]:
infercnv_dir = '/depot/natallah/data/Luopin/Metastasis_single_cell/Data/InferCNVpy_results_v2/'

In [ ]:
cancer_ad_list = []
all_mal_cell = 0
high_cnv_mal_cell = 0
for h5 in all_h5_files:
    
    if h5.__contains__('ntegrated'):
        continue
    if not h5.__contains__('COAD'):
        continue
    print('*'*20)
    print(h5)
    # continue
    tmp_ad = sc.read_h5ad(data_dir+h5)
    tmp_ad.X = tmp_ad.raw.X
    tmp_ad.raw = None
    
    if h5.__contains__('2098-Colorectalcancer'):
        tmp_ad.obs_names = tmp_ad.obs['Cell']

    # Convert var_names from Categorical to regular string index
    tmp_ad.var_names = tmp_ad.var_names.astype(str)

    tmp_ad.var_names_make_unique()
    tmp_ad.obs_names_make_unique()
    
    original_cell_num = tmp_ad.n_obs
    all_mal_cell += original_cell_num
    print(f'Loaded {tmp_ad.n_obs} cells...')
    
    # get the malignant cell indices
    proj_name = h5.strip('.h5ad')
    indices_file = infercnv_dir+proj_name+'.barcodes.txt'
    malignant_indices = pd.read_csv(indices_file, header=None, skiprows=2)
    print(f'Loaded {len(malignant_indices)} malignant cells...')
                    
    shared_cells = list(set(tmp_ad.obs_names).intersection(set(malignant_indices[0].values)))  
    tmp_ad = tmp_ad[shared_cells]
    
    high_cnv_mal_cell += tmp_ad.n_obs
    
    print(f'Kept {len(shared_cells)/original_cell_num*100:.2f}% malignant cells...')
    
    # tmp_ad.X = tmp_ad.raw.X.copy()
    cancer_ad_list.append(tmp_ad)
    
    # print(cancer_ad_list[-1])
    # display(tmp_ad.to_df())

print('-'*20)    
print(f'Kept {high_cnv_mal_cell/all_mal_cell*100:.2f}% malignant cells...')

In [ ]:
memory_usgae()

In [ ]:
combined_ad = ann.concat(cancer_ad_list, join="inner", axis=0)
combined_ad

In [ ]:
combined_ad.raw = combined_ad.copy()
combined_ad.raw.shape

In [ ]:
combined_ad = reprocess_all(combined_ad)

### Univfy Labels

In [ ]:
combined_ad.obs["Final_histological_subtype"].value_counts()

In [ ]:
combined_ad.obs["Final_histological_subtype_backup"] = combined_ad.obs["Final_histological_subtype"].copy()


In [ ]:
def unify_crc_histology(val):
    val = str(val).lower()
    if "mucinous" in val and "neuroendocrine" in val:
        subtype = "Mucinous neuroendocrine carcinoma"
    elif "mucinous" in val:
        subtype = "Mucinous adenocarcinoma"
    elif "neuroendocrine" in val:
        subtype = "Neuroendocrine tumor"
    elif "medullary" in val:
        subtype = "Medullary carcinoma"
    elif "adenocarcinoma" in val:
        subtype = "Adenocarcinoma"
    elif "unspecified" in val:
        subtype = "Unspecified"
    else:
        subtype = "Unspecified"
    
    return f"COAD: {subtype}"


combined_ad.obs["Final_histological_subtype"] = combined_ad.obs["Final_histological_subtype_backup"].apply(unify_crc_histology)
combined_ad.obs['Final_histological_subtype'].value_counts()

In [ ]:
combined_ad.obs['Final_molecular_subtype'].value_counts()

In [ ]:
combined_ad.obs["Final_molecular_subtype_backup"] = combined_ad.obs["Final_molecular_subtype"].copy()


In [ ]:
def unify_crc_molecular(val):
    val = str(val).strip().lower().replace("coad:", "").strip()
    
    if val in {"mmrp", "mss"}:
        subtype = "MMRp"
    elif val in {"mmrd", "msi-high"}:
        subtype = "MMRd"
    elif "unspecified" in val or 'unknown' in val:
        subtype = "Unspecified"
    else:
        subtype = val.capitalize()
    
    return f"COAD: {subtype}"


combined_ad.obs["Final_molecular_subtype"] = combined_ad.obs["Final_molecular_subtype_backup"].apply(unify_crc_molecular)
combined_ad.obs['Final_molecular_subtype'].value_counts()

In [ ]:
combined_ad.obs["Primary_or_Metastatic"].value_counts()

In [ ]:
def map_primary_metastatic(val):
    val = str(val).strip().lower()
    if val == "m0":
        return "Primary"
    else:
        return val.capitalize()  # keeps values like 'Metastatic', 'Primary', etc.

combined_ad.obs["Primary_or_Metastatic"] = combined_ad.obs["Primary_or_Metastatic"].apply(map_primary_metastatic)


In [ ]:
for obs in ['Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue']:
    sc.pl.umap(combined_ad, color=obs)

### Harmony integration

In [ ]:
combined_ad

In [ ]:
Z = harmonize(combined_ad.obsm['X_pca'], combined_ad.obs, batch_key = ['Project_ID'])


In [ ]:
combined_ad.obsm['X_pca_harmony'] = Z


In [ ]:
sc.pp.neighbors(combined_ad, n_neighbors=15, use_rep='X_pca_harmony')
sc.tl.umap(combined_ad)

In [ ]:
for obs in ['Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue']:
    sc.pl.umap(combined_ad, color=obs)

In [ ]:
combined_ad.to_df()

In [ ]:
combined_ad.obs["Final_patient_age_backup"] = combined_ad.obs["Final_patient_age"]

def clean_patient_age(age):
    if pd.isna(age):
        return np.nan
    age = str(age).strip()
    if age.lower() == "unknown":
        return np.nan
    elif "-" in age:
        # Convert age ranges like '46-50' to their midpoint
        parts = age.split("-")
        try:
            return int((int(parts[0]) + int(parts[1])) / 2)
        except:
            return np.nan
    else:
        try:
            return int(age)
        except:
            return np.nan

# Apply cleaning
combined_ad.obs["Final_patient_age"] = combined_ad.obs["Final_patient_age_backup"].apply(clean_patient_age)
combined_ad.obs["Final_patient_age_backup"] =combined_ad.obs["Final_patient_age_backup"].astype(str)

In [ ]:

combined_ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/COAD_integrated.harmony.h5ad', compression='gzip')
